# DDoS LSTM — LangGraph Analyzer (Google Colab / Groq)

Ovaj notebook je adaptacija `langraph.py` + `langraph_test.py` iz Docker okruženja.
**Jedina razlika:** umesto lokalnog Ollama servera, koristi se **Groq API** (cloud inference, besplatno, veoma brzo).

---

## Šta ovaj notebook radi

Pokreće **LangGraph graf** koji:
1. Učitava `test_metrics.json` iz projekta
2. Poredi sa prethodnim run-om (ako postoji `test_metrics_prev.json`)
3. Analizira per-class metrike i matricu konfuzije pomoću LLM-a
4. Sintetizuje preporuke za poboljšanje modela
5. Donosi odluku: `SUGGEST_ONLY` / `SCAN_FIRST` / `RETRAIN`
6. (Opcionalno) Skenira relevantne fajlove i predlaže hyperparametre

## Priprema pre pokretanja

1. Uploaduj ceo direktorijum projekta u Google Drive
2. U ćeliji **[2] Konfiguracija** postavi putanju do projekta i Groq API ključ
3. Pokreni sve ćelije redom (Runtime → Run all)

> **Napomena o retrain-u:** `node_trigger_retrain` pokreće `torch_nn.py` subprocesom.  
> U Colab okruženju to radi samo ako je Python fajl dostupan na disku.  
> Ako treniraš model lokalno, možeš preskočiti confirmation ćeliju.


## [1] Instalacija zavisnosti

Instalira sve što je potrebno. Ovo traje ~1 minutu pri prvom pokretanju.

In [ ]:
!pip install -q langgraph langchain-core groq 

## [2] Konfiguracija

**Ovde se postavljaju svi parametri potrebni za pokretanje.**

- `GROQ_API_KEY` — tvoj Groq ključ sa https://console.groq.com/keys
- `PROJECT_ROOT` — putanja do foldera projekta na Google Drive-u  
  *(nakon što montiraš Drive u ćeliji ispod, putanja je oblika `/content/drive/MyDrive/tvoj_folder`)*
- `METRICS_FILE` — ime fajla sa metrikama, relativno od `PROJECT_ROOT`
- `GROQ_MODEL` — preporučeni modeli `openai/gpt-oss-120b` ili `llama-3.3-70b-versatile` 

In [ ]:
import os
import sys
from google.colab import drive, userdata



# ─── PO POTREBI PROMENITI ──────────────────────────────────────────

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
PROJECT_ROOT  = "/content/drive/MyDrive/PrimenaAI/ddos"  # putanja do projekta
METRICS_FILE  = "results/test_metrics.json"  # relativno od PROJECT_ROOT
GROQ_MODEL    = "llama-3.3-70b-versatile"       # ili "openai/gpt-oss-120b"
TORCH_FILE = 'torch_nn.py'

# ───────────────────────────────────────────────────────────────────────────

drive.mount('/content/drive')

torch_full_path = os.path.join(PROJECT_ROOT, TORCH_FILE)
if os.path.exists(torch_full_path):
    print(f" Torch file found: {torch_full_path}")
else:
    print(f" Torch file NOT found: {torch_full_path}")
    print("  Proveri putanju u ćeliji [2] ili uploaduj projekat na Drive.")


metrics_full_path = os.path.join(PROJECT_ROOT, METRICS_FILE)
if os.path.exists(metrics_full_path):
    print(f" Metrics file found: {metrics_full_path}")
else:
    print(f" Metrics file NOT found: {metrics_full_path}")
    print("  Proveri putanju u ćeliji [2] ili uploaduj projekat na Drive.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Dodaj PROJECT_ROOT u Python path da bi import-i (attacks.py itd.) radili
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Prebaci radni direktorijum na root projekta
os.chdir(PROJECT_ROOT)



print(f"Project root: {os.getcwd()}")
print(f"Metrics file: {METRICS_FILE}")
print(f"Groq model:   {GROQ_MODEL}")
print(f"API key set:  {'YES' if GROQ_API_KEY != 'gsk_YOUR_KEY_HERE' else 'NO — POSTAVI PRAVI KLJUC!'}")

## [3] Groq klijent — zamena za Ollama

Ovo je direktna zamena za `ollama_client.py`. Svi promptovi su identični originalu —  
jedino što se `ollama_generate()` sad poziva preko Groq API-ja umesto lokalnog Ollama servera.

Funkcije koje su bile u `ollama_client.py` (`generate_attack_descriptions`, `generate_attack_analysis`, itd.) su ovde reimplementirane sa Groq backendom.

In [ ]:
import json
import logging
import re
import random
from groq import AsyncGroq

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

# Jedan shared klijent za ceo notebook (ekvivalent get_http_client() iz ollama_client.py)
_groq_client: AsyncGroq | None = None

def get_groq_client() -> AsyncGroq:
    global _groq_client
    if _groq_client is None:
        _groq_client = AsyncGroq(api_key=os.environ["GROQ_API_KEY"])
    return _groq_client


def _extract_json(raw: str) -> str:
    """
    Izvlači JSON iz odgovora koji može biti umotan u markdown code blok.
    Identično originalnoj funkciji iz ollama_client.py.
    """
    match = re.search(r"```(?:json)?\s*([\s\S]+?)```", raw)
    if match:
        return match.group(1).strip()
    return raw.strip()


async def ollama_generate(prompt: str, system: str = "") -> str:
    """
    Drop-in zamena za ollama_generate() iz ollama_client.py.
    Isti interfejs, isti parametri — samo backend je Groq umesto Ollama.
    """
    client = get_groq_client()
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    try:
        response = await client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            max_tokens=2048,
            temperature=0.3,  # niža temperatura = konzistentniji JSON output
        )
        return response.choices[0].message.content or ""
    except Exception as e:
        logger.error(f"Groq API error: {e}")
        raise RuntimeError(f"Groq not available: {e}")


async def generate_attack_descriptions(attacks_source: str) -> dict:
    """
    Identičan prompt kao u ollama_client.py — generiše opise tipova napada
    na osnovu source koda attacks.py.
    """
    prompt = f"""Below is the source code of a Python file that defines DDoS attack types used to train a neural network classifier.

{attacks_source}

    For each attack type defined in this file, provide a JSON object where:
    - Each key is the attack type name (snake_case, exactly as used in the code)
    - Each value is an object with:
    - "description": 2-3 sentence technical description of this attack, how it works, and what makes it distinct
    - "characteristics": list of 3-5 short strings describing key traffic features (e.g. "High UDP packet rate", "Spoofed source IPs")

    Respond ONLY with valid JSON. No markdown, no explanation."""

    system = (
        "You are a network security expert. "
        "You analyze DDoS attack implementations and explain them clearly and technically. "
        "You respond ONLY with valid JSON."
    )

    raw = await ollama_generate(prompt, system)
    clean = _extract_json(raw)

    try:
        result = json.loads(clean)
    except json.JSONDecodeError as e:
        logger.error(f"Groq returned invalid JSON for attack descriptions: {e}")
        raise ValueError(f"Groq did not return valid JSON: {e}")

    if not isinstance(result, dict):
        raise ValueError("Expected a JSON object, got something else.")

    return result


# Testiranje klijenta
import asyncio

async def _test_groq():
    resp = await ollama_generate("Reply with exactly: {\"status\": \"ok\"}", system="You respond ONLY with valid JSON.")
    print(f"Groq test response: {resp[:100]}")
    print(" Groq klijent radi ispravno")

await _test_groq()

## [4] Helper funkcije i konstante

Sve pomoćne funkcije iz `langraph.py` — formateri, delta računanje, validacija hyperparametara.


In [ ]:
from pathlib import Path
from typing import TypedDict, Optional

# ─── Paths ──────────────────────────────────────────────────────────────────

# U Colabu se putanje računaju relativno od PROJECT_ROOT (os.chdir u ćeliji [2])
METRICS_PATH      = Path(METRICS_FILE)
PREV_METRICS_PATH = Path(METRICS_FILE.replace(".json", "_prev.json"))

# ─── Pragovi za decision logiku ─────────────────────────────────────────────
DECISION_RETRAIN_MCC = 0.75
DECISION_SCAN_MCC    = 0.85

# ─── Mapiranje slabe klase → fajlovi za skeniranje ─────────────────────────
ALWAYS_SCAN = ["hyperparam.py", "config.py"]
DEFAULT_DATASET_PATH = "output/1d.csv"

CLASS_TO_FILES: dict = {
    "udp_flood_large":       ["attacks.py", "dataset_generator.py"],
    "udp_flood_mixed":       ["attacks.py", "dataset_generator.py"],
    "dns_amplification":     ["attacks.py", "dataset_generator.py"],
    "ntp_amplification":     ["attacks.py", "dataset_generator.py"],
    "syn_flood":             ["attacks.py", "windowing.py"],
    "ack_flood":             ["attacks.py", "windowing.py"],
    "icmp_flood":            ["attacks.py", "dataset_generator.py"],
    "subnet_carpet_bombing": ["attacks.py", "dataset_generator.py"],
    "normal":                ["dataset_generator.py", "normal.py"],
}

SCAN_MAX_CHARS_PER_FILE = 3000
SCAN_SKIP_DIRS: set = {"__pycache__", "venv", ".venv", ".git", ".idea"}

# ─── Prostor pretrage hyperparametara ───────────────────────────────────────
_HYPERPARAMETER_SPACE = {
    "hidden_size":   {"type": "choice", "values": [64, 128, 256]},
    "num_layers":    {"type": "choice", "values": [1, 2]},
    "dropout":       {"type": "range",  "min": 0.1, "max": 0.4},
    "learning_rate": {"type": "range",  "min": 1e-4, "max": 1e-2},
    "seq_len":       {"type": "choice", "values": [20, 30, 50]},
}


# ─── State TypedDict ─────────────────────────────────────────────────────────

class MetricsState(TypedDict):
    classification_report: dict
    confusion_matrix: list
    mcc_score: float
    roc_auc_scores: dict
    class_labels: list
    previous_metrics: dict
    metrics_delta: dict
    regression_detected: bool
    per_class_analysis: str
    confusion_analysis: str
    attack_descriptions: dict
    weak_classes: list
    recommendations: dict
    summary: str
    decision: str
    scanned_files: dict
    proposed_hyperparams: dict
    human_confirmed: bool
    retrain_triggered: bool
    retrain_command: str
    dataset_path: str
    model_name: str


# ─── Helper: formateri ──────────────────────────────────────────────────────

def _format_per_class_metrics(report: dict, roc_auc: dict, mcc: float) -> str:
    lines = [f"Overall MCC: {mcc:.4f}\n"]
    for cls, m in report.items():
        if not isinstance(m, dict):
            continue
        lines.append(
            f"  {cls}:\n"
            f"    precision={m.get('precision', 0):.3f}  "
            f"recall={m.get('recall', 0):.3f}  "
            f"f1={m.get('f1-score', 0):.3f}  "
            f"support={int(m.get('support', 0))}  "
            f"roc_auc={roc_auc.get(cls, 0):.3f}"
        )
    return "\n".join(lines)


def _format_confusion_matrix(matrix: list, labels: list) -> str:
    col_width = max(len(l) for l in labels) + 2
    header = " " * col_width + "  ".join(f"{l:>{col_width}}" for l in labels)
    rows = []
    for i, row in enumerate(matrix):
        row_str = f"{labels[i]:>{col_width}}  " + "  ".join(f"{v:>{col_width}}" for v in row)
        rows.append(row_str)
    return header + "\n" + "\n".join(rows)


def _safe_parse_json(raw: str, fallback: dict) -> dict:
    clean = _extract_json(raw)
    try:
        return json.loads(clean)
    except json.JSONDecodeError as e:
        logger.warning(f"JSON parsing failed: {e} | Raw snippet: {raw[:200]}")
        return fallback


# ─── Helper: delta računanje ─────────────────────────────────────────────────

def _compute_delta(current: dict, previous: dict) -> tuple:
    if not previous:
        return {
            "mcc_delta": None,
            "macro_f1_delta": None,
            "per_class_f1_delta": {},
            "note": "No previous run found — first baseline.",
        }, False

    prev_mcc  = previous.get("mcc_score", 0.0)
    curr_mcc  = current["mcc_score"]
    mcc_delta = round(curr_mcc - prev_mcc, 4)

    prev_f1  = previous.get("summary", {}).get("macro_f1", 0.0)
    curr_f1  = current.get("summary", {}).get("macro_f1", 0.0)
    f1_delta = round(curr_f1 - prev_f1, 4) if curr_f1 and prev_f1 else None

    prev_report = previous.get("classification_report", {})
    curr_report = current["classification_report"]
    per_class_delta: dict = {}

    for cls, curr_m in curr_report.items():
        if not isinstance(curr_m, dict):
            continue
        prev_m = prev_report.get(cls, {})
        if isinstance(prev_m, dict):
            diff = round(curr_m.get("f1-score", 0.0) - prev_m.get("f1-score", 0.0), 4)
            per_class_delta[cls] = diff

    regression = (
        mcc_delta < 0
        or (f1_delta is not None and f1_delta < 0)
        or any(v < -0.03 for v in per_class_delta.values())
    )

    delta = {
        "mcc_delta": mcc_delta,
        "macro_f1_delta": f1_delta,
        "per_class_f1_delta": per_class_delta,
    }

    return delta, regression


# ─── Helper: file search i hyperparams ──────────────────────────────────────

def _find_file_recursive(filename: str, root: Path) -> Path | None:
    for path in root.rglob(filename):
        if any(part in SCAN_SKIP_DIRS for part in path.parts):
            continue
        return path
    return None


def _validate_hyperparams(proposed: dict) -> dict:
    validated = {}
    for param, spec in _HYPERPARAMETER_SPACE.items():
        raw = proposed.get(param)
        if raw is None:
            logger.warning(f"  Missing proposed value for '{param}' — skipping.")
            continue
        if spec["type"] == "choice":
            if raw not in spec["values"]:
                closest = min(spec["values"], key=lambda v: abs(v - raw))
                logger.warning(f"  '{param}' value {raw} not in {spec['values']} — clamped to {closest}.")
                validated[param] = closest
            else:
                validated[param] = raw
        elif spec["type"] == "range":
            clamped = max(spec["min"], min(spec["max"], float(raw)))
            validated[param] = round(clamped, 6)
    return validated


print(" Helper funkcije i konstante učitane")

## [5] LangGraph čvorovi (nodes)

Svi čvorovi su identični kao u `langraph.py`. Jedina razlika je što se pozivi
na `from ollama_client import ...` zamenjuju funkcijama definisanim u ćeliji [4].

**Redosled izvršavanja grafa:**
```
load_and_compare → load_attack_descriptions → analyze_per_class
    → analyze_confusion → synthesize → decision
        ├── SUGGEST_ONLY → END
        ├── SCAN_FIRST   → scan_codebase → propose_hyperparams → END
        └── RETRAIN      → propose_hyperparams → END
```

In [ ]:
# ─── Node 0: Učitavanje i poređenje metrika ──────────────────────────────────

async def node_load_and_compare_metrics(state: MetricsState) -> dict:
    """
    Učitava prethodni run iz *_prev.json i računa delta vrednosti.
    Ne poziva Groq — čista Python logika.
    """
    logger.info("[Node 0] load_and_compare_metrics: start")

    previous: dict = {}
    if PREV_METRICS_PATH.exists():
        try:
            with open(PREV_METRICS_PATH, "r") as f:
                previous = json.load(f)
            logger.info(f"Previous metrics loaded: MCC={previous.get('mcc_score', 'N/A')}")
        except Exception as e:
            logger.warning(f"Could not load previous metrics: {e}")
    else:
        logger.info("No previous metrics file — first baseline run.")

    current_as_dict = {
        "mcc_score":             state["mcc_score"],
        "classification_report": state["classification_report"],
        "summary":               {},
    }
    delta, regression = _compute_delta(current_as_dict, previous)

    if previous and delta.get("mcc_delta") is not None:
        sign = "▲" if delta["mcc_delta"] >= 0 else "▼"
        logger.info(f"  MCC delta: {sign} {abs(delta['mcc_delta']):.4f}")

    logger.info("[Node 0] load_and_compare_metrics: done")
    return {
        "previous_metrics":    previous,
        "metrics_delta":       delta,
        "regression_detected": regression,
    }


# ─── Node 1: Učitavanje opisa napada ─────────────────────────────────────────

async def node_load_attack_descriptions(state: MetricsState) -> dict:
    """
    Čita attacks.py i koristi Groq da generiše tehničke opise svakog tipa napada.
    Ako attacks.py nije dostupan, vraća prazan rečnik — graf nastavlja normalano.
    """
    logger.info("[Node 1] load_attack_descriptions: start")

    attacks_path = _find_file_recursive("attacks.py", Path("."))
    if attacks_path is None:
        logger.warning("  attacks.py not found — skipping attack descriptions.")
        return {"attack_descriptions": {}}

    try:
        source = attacks_path.read_text(encoding="utf-8")
        descriptions = await generate_attack_descriptions(source)
        logger.info(f"[Node 1] done | {len(descriptions)} attacks described")
        return {"attack_descriptions": descriptions}
    except Exception as e:
        logger.warning(f"  Failed to generate attack descriptions: {e}")
        return {"attack_descriptions": {}}


# ─── Node 2: Per-class analiza ───────────────────────────────────────────────

async def node_analyze_per_class(state: MetricsState) -> dict:
    """Analizira precision, recall, f1-score i ROC-AUC po klasi. Identičan prompt kao u originalu."""
    logger.info("[Node 2] analyze_per_class: start")

    metrics_str  = _format_per_class_metrics(state["classification_report"], state["roc_auc_scores"], state["mcc_score"])
    delta        = state.get("metrics_delta", {})
    has_delta    = delta.get("mcc_delta") is not None
    descriptions = state.get("attack_descriptions", {})

    attack_ctx = ""
    if descriptions:
        lines = ["ATTACK TYPE DEFINITIONS (from training data generator):"]
        for cls in state["class_labels"]:
            info = descriptions.get(cls)
            if info:
                chars = ", ".join(info.get("characteristics", []))
                lines.append(f"  {cls}:")
                lines.append(f"    Description: {info.get('description', '')}")
                lines.append(f"    Key characteristics: {chars}")
        attack_ctx = "\n".join(lines) + "\n"

    if has_delta:
        sign = "improved" if delta["mcc_delta"] >= 0 else "regressed"
        delta_ctx = f"\nCOMPARISON WITH PREVIOUS RUN:\n  MCC delta: {delta['mcc_delta']:+.4f} ({sign})\n"
        per_class = delta.get("per_class_f1_delta", {})
        if per_class:
            regressed = [(c, v) for c, v in per_class.items() if v < -0.03]
            improved  = [(c, v) for c, v in per_class.items() if v >  0.03]
            if regressed:
                delta_ctx += "  Classes with notable F1 regression (>0.03): " + ", ".join(f"{c} ({v:+.3f})" for c, v in regressed) + "\n"
            if improved:
                delta_ctx += "  Classes with notable F1 improvement (>0.03): " + ", ".join(f"{c} ({v:+.3f})" for c, v in improved) + "\n"
    else:
        delta_ctx = "\nCOMPARISON WITH PREVIOUS RUN: Not available (first run or no baseline).\n"

    prompt = f"""You are analyzing per-class performance of a DDoS traffic classification LSTM model.

    The model classifies network traffic windows into 9 classes:
    - normal traffic
    - 8 DDoS attack types: udp_flood_large, dns_amplification, subnet_carpet_bombing,
    syn_flood, icmp_flood, udp_flood_mixed, ntp_amplification, ack_flood

    Per-class metrics:
    {metrics_str}
    {delta_ctx}
    {attack_ctx}

    Thresholds for "weak": f1-score < 0.80 OR recall < 0.75 OR roc_auc < 0.85

    Respond ONLY with a valid JSON object (no markdown, no explanation outside JSON):
    {{
    "weak_classes": ["list of class names that fall below thresholds"],
    "analysis": "Technical paragraph (4-6 sentences) explaining WHY these specific classes underperform."
    }}"""

    system = (
        "You are an ML engineer specializing in network traffic classification. "
        "You analyze model performance metrics and identify root causes of underperformance. "
        "You respond ONLY with valid JSON."
    )

    raw    = await ollama_generate(prompt, system)
    result = _safe_parse_json(raw, fallback={"weak_classes": [], "analysis": raw[:500]})

    logger.info(f"[Node 2] done | weak_classes={result.get('weak_classes', [])}")
    return {
        "weak_classes":      result.get("weak_classes", []),
        "per_class_analysis": result.get("analysis", raw[:500]),
    }


# ─── Node 3: Analiza konfuzione matrice ──────────────────────────────────────

async def node_analyze_confusion(state: MetricsState) -> dict:
    """Analizira matricu konfuzije i identifikuje obrasce grešaka."""
    logger.info("[Node 3] analyze_confusion: start")

    matrix_str = _format_confusion_matrix(state["confusion_matrix"], state["class_labels"])
    weak_ctx = (
        f"Previously identified weak classes: {', '.join(state['weak_classes'])}"
        if state["weak_classes"] else "No weak classes identified yet."
    )

    prompt = f"""You are analyzing a confusion matrix from a DDoS traffic classification model.

    Classes (in order): {", ".join(state["class_labels"])}

    Confusion Matrix (rows = actual class, columns = predicted class):
    {matrix_str}

    {weak_ctx}

    Identify the most significant misclassification patterns.

    Respond ONLY with a valid JSON object:
    {{
    "confusion_patterns": "Technical paragraph (4-6 sentences) describing which classes are most often confused with each other, likely reasons, and any systematic biases.",
    "top_confusions": [
        {{"actual": "class_name", "predicted": "class_name", "count": 123}}
    ]
    }}"""

    system = (
        "You are an ML engineer specializing in network traffic classification. "
        "You respond ONLY with valid JSON."
    )

    raw    = await ollama_generate(prompt, system)
    result = _safe_parse_json(raw, fallback={"confusion_patterns": raw[:500], "top_confusions": []})

    logger.info("[Node 3] analyze_confusion: done")
    return {"confusion_analysis": result.get("confusion_patterns", raw[:500])}


# ─── Node 4: Sinteza preporuka ───────────────────────────────────────────────

async def node_synthesize_recommendations(state: MetricsState) -> dict:
    """Sintetizuje konkretne preporuke za poboljšanje (data, model, training)."""
    logger.info("[Node 4] synthesize_recommendations: start")

    weak_str   = ", ".join(state["weak_classes"]) if state["weak_classes"] else "none identified"
    delta      = state.get("metrics_delta", {})
    has_delta  = delta.get("mcc_delta") is not None
    regression = state.get("regression_detected", False)

    trend_ctx = (
        f"  Trend vs previous run: MCC {delta['mcc_delta']:+.4f}, "
        f"macro F1 {delta.get('macro_f1_delta', 0) or 0:+.4f} "
        f"({'REGRESSION DETECTED' if regression else 'no regression'})\n"
        if has_delta else "  Trend vs previous run: N/A (first run)\n"
    )

    prompt = f"""You are synthesizing actionable improvement recommendations for a DDoS LSTM classifier.

    CONTEXT:
    - Model: 2-layer LSTM with attention, sliding window of 20 samples x 12 features
    - Features: packet_size, packets_per_second, bytes_per_second, src_port, dst_port,
    protocol, tcp_flags, ttl, inter_arrival_time, flow_duration, unique_src_ips, unique_dst_ports
    - Data: synthetic traffic with Gaussian noise, transition period blending, minority class oversampling
    - Training: CrossEntropyLoss with inverse-frequency class weights, Adam optimizer, ReduceLROnPlateau

    ANALYSIS RESULTS:
    - Overall MCC: {state['mcc_score']:.4f}
    {trend_ctx}
    - Weak classes: {weak_str}
    - Per-class analysis: {state['per_class_analysis']}
    - Confusion matrix analysis: {state['confusion_analysis']}

    Respond ONLY with a valid JSON object:
    {{
    "data_generation": ["3-4 specific recommendations for improving the synthetic data generator"],
    "model":           ["3-4 specific recommendations for LSTM architecture or hyperparameters"],
    "training":        ["2-3 specific recommendations for training strategy"],
    "summary": "2-3 sentence executive summary."
    }}"""

    system = (
        "You are a senior ML engineer. You provide specific, implementable recommendations "
        "grounded in the actual model architecture and data pipeline described. "
        "You respond ONLY with valid JSON."
    )

    raw    = await ollama_generate(prompt, system)
    result = _safe_parse_json(raw, fallback={"data_generation": [], "model": [], "training": [], "summary": raw[:300]})

    # Normalizacija — Groq/Ollama ponekad vrati list of dicts umesto list of strings
    for key in ("data_generation", "model", "training"):
        items = result.get(key, [])
        result[key] = [
            item.get("recommendation") or item.get("text") or str(item)
            if isinstance(item, dict) else str(item)
            for item in items
        ]

    logger.info("[Node 4] synthesize_recommendations: done")
    return {
        "recommendations": {
            "data_generation": result.get("data_generation", []),
            "model":           result.get("model", []),
            "training":        result.get("training", []),
        },
        "summary": result.get("summary", ""),
    }


# ─── Node 5: Decision (bez LLM-a) ────────────────────────────────────────────

def node_decision(state: MetricsState) -> dict:
    """Deterministički određuje sledeći korak na osnovu MCC i regresije."""
    logger.info("[Node 5] decision: start")
    mcc        = state["mcc_score"]
    regression = state.get("regression_detected", False)

    if mcc < DECISION_RETRAIN_MCC:
        decision = "RETRAIN"
    elif mcc < DECISION_SCAN_MCC or regression:
        decision = "SCAN_FIRST"
    else:
        decision = "SUGGEST_ONLY"

    logger.info(f"[Node 5] decision: MCC={mcc:.4f} | regression={regression} | decision={decision}")
    return {"decision": decision}


def _route_after_decision(state: MetricsState) -> str:
    return state["decision"]


# ─── Node 6: Skeniranje koda ─────────────────────────────────────────────────

async def node_scan_codebase(state: MetricsState) -> dict:
    """Rekurzivno skenira relevantne fajlove na osnovu slabih klasa."""
    logger.info("[Node 6] scan_codebase: start")

    files_to_scan: set = set(ALWAYS_SCAN)
    for cls in state.get("weak_classes", []):
        files_to_scan.update(CLASS_TO_FILES.get(cls, []))

    logger.info(f"  Target files: {sorted(files_to_scan)}")
    scanned: dict = {}
    root = Path(".")

    for filename in sorted(files_to_scan):
        filepath = _find_file_recursive(filename, root)
        if filepath is None:
            scanned[filename] = f"[FILE NOT FOUND: {filename}]"
            logger.warning(f"  Not found: {filename}")
            continue
        try:
            content = filepath.read_text(encoding="utf-8")
            if len(content) > SCAN_MAX_CHARS_PER_FILE:
                content = content[:SCAN_MAX_CHARS_PER_FILE] + f"\n... [truncated]"
            rel_key = str(filepath.relative_to(root))
            scanned[rel_key] = content
            logger.info(f"  Read {rel_key}: {len(content)} chars")
        except Exception as e:
            scanned[filename] = f"[READ ERROR: {e}]"

    logger.info(f"[Node 6] done | {len(scanned)} files scanned")
    return {"scanned_files": scanned}


# ─── Node 7: Predlog hyperparametara ─────────────────────────────────────────

async def node_propose_hyperparams(state: MetricsState) -> dict:
    """Poziva Groq da predloži konkretne hyperparametre u okviru definisanog prostora."""
    logger.info("[Node 7] propose_hyperparams: start")

    space_lines = []
    for param, spec in _HYPERPARAMETER_SPACE.items():
        if spec["type"] == "choice":
            space_lines.append(f"  - {param}: choose ONE from {spec['values']}")
        else:
            space_lines.append(f"  - {param}: float in range [{spec['min']}, {spec['max']}]")
    space_str = "\n".join(space_lines)

    scanned = state.get("scanned_files", {})
    scan_ctx = "No scanned files available."
    if scanned:
        scan_ctx = "Scanned project files (relevant excerpts):\n"
        for fname, content in scanned.items():
            if not content.startswith("["):
                scan_ctx += f"\n--- {fname} ---\n{content[:800]}\n"

    prompt = f"""You are tuning hyperparameters for a DDoS detection LSTM model.

        ANALYSIS SUMMARY:
        - MCC score: {state['mcc_score']:.4f}
        - Decision: {state.get('decision', 'N/A')}
        - Weak classes: {', '.join(state.get('weak_classes', [])) or 'none'}
        - Key findings: {state.get('per_class_analysis', '')[:400]}
        - Recommendations: {str(state.get('recommendations', {}))[:400]}

        {scan_ctx}

        HYPERPARAMETER SPACE (you MUST stay within these bounds):
        {space_str}

        Respond ONLY with a valid JSON object, example:
        {{
        "hidden_size": 256,
        "num_layers": 2,
        "dropout": 0.25,
        "learning_rate": 0.001,
        "seq_len": 30,
        "reasoning": "Short explanation (2-3 sentences) why these values address the identified weak classes."
        }}"""

    system = (
        "You are an ML engineer specializing in LSTM hyperparameter optimization. "
        "You always respect the given hyperparameter space boundaries. "
        "You respond ONLY with valid JSON."
    )

    raw    = await ollama_generate(prompt, system)
    result = _safe_parse_json(raw, fallback={
        "hidden_size": 128, "num_layers": 2, "dropout": 0.3,
        "learning_rate": 0.001, "seq_len": 30,
        "reasoning": "Fallback defaults — LLM response could not be parsed.",
    })

    reasoning = result.pop("reasoning", "")
    validated = _validate_hyperparams(result)
    validated["_reasoning"] = reasoning

    logger.info(f"[Node 7] done | proposed={validated}")
    return {"proposed_hyperparams": validated}


print(" Svi LangGraph čvorovi definisani")

## [6] Izgradnja LangGraph grafa

Identična struktura kao `build_analyzer_graph()` u `langraph.py`.  
`node_human_confirm` i `node_trigger_retrain` su **van grafa** — pozivaju se ručno u ćeliji [9] kao u originalnom `langraph_test.py`.

In [ ]:
from langgraph.graph import StateGraph, END

def build_analyzer_graph():
    graph = StateGraph(MetricsState)

    graph.add_node("load_and_compare",        node_load_and_compare_metrics)
    graph.add_node("load_attack_descriptions", node_load_attack_descriptions)
    graph.add_node("analyze_per_class",        node_analyze_per_class)
    graph.add_node("analyze_confusion",        node_analyze_confusion)
    graph.add_node("synthesize",               node_synthesize_recommendations)
    graph.add_node("decision",                 node_decision)
    graph.add_node("scan_codebase",            node_scan_codebase)
    graph.add_node("propose_hyperparams",      node_propose_hyperparams)

    graph.set_entry_point("load_and_compare")
    graph.add_edge("load_and_compare",         "load_attack_descriptions")
    graph.add_edge("load_attack_descriptions", "analyze_per_class")
    graph.add_edge("analyze_per_class",        "analyze_confusion")
    graph.add_edge("analyze_confusion",        "synthesize")
    graph.add_edge("synthesize",               "decision")

    graph.add_conditional_edges(
        "decision",
        _route_after_decision,
        {
            "SUGGEST_ONLY": END,
            "SCAN_FIRST":   "scan_codebase",
            "RETRAIN":      "propose_hyperparams",
        },
    )

    graph.add_edge("scan_codebase",    "propose_hyperparams")
    graph.add_edge("propose_hyperparams", END)

    return graph.compile()

analyzer_graph = build_analyzer_graph()
print(" LangGraph graf kompajliran")

## [7] Ispis rezultata

Helper funkcije za formatiran prikaz u Colabu — ekvivalent `print_results()` iz `langraph_test.py`.

In [ ]:
def separator(char="=", width=65):
    print(char * width)

def section(title: str):
    print()
    separator()
    print(f"  {title}")
    separator()

def print_list(items: list, indent: int = 4):
    for i, item in enumerate(items, 1):
        text = item.get("recommendation") or item.get("text") or str(item) if isinstance(item, dict) else str(item)
        words = text.split()
        line, lines = [], []
        for word in words:
            if len(" ".join(line + [word])) > 90:
                lines.append(" ".join(line))
                line = [word]
            else:
                line.append(word)
        if line:
            lines.append(" ".join(line))
        print(f"{' ' * indent}{i}. {lines[0]}")
        for cont in lines[1:]:
            print(" " * (indent + 4) + cont)

def print_attack_descriptions(descriptions: dict):
    if not descriptions:
        print("  (Attack descriptions not available)")
        return
    for attack_name, info in descriptions.items():
        print(f"\n  {attack_name}")
        print(f"  {'─' * 40}")
        print(f"    {info.get('description', '—')}")
        chars = info.get("characteristics", [])
        if chars:
            print(f"    Key traits: {' | '.join(chars)}")

def print_results(state: dict):
    section("LANGGRAPH ANALYSIS RESULTS")

    delta = state.get("metrics_delta", {})
    if delta.get("mcc_delta") is not None:
        print("\n  Comparison with previous run:")
        mcc_d = delta["mcc_delta"]
        f1_d  = delta.get("macro_f1_delta")
        reg   = state.get("regression_detected", False)
        print(f"    MCC:      {'▲' if mcc_d >= 0 else '▼'} {mcc_d:+.4f}")
        if f1_d is not None:
            print(f"    Macro F1: {'▲' if f1_d >= 0 else '▼'} {f1_d:+.4f}")
        per_cls = delta.get("per_class_f1_delta", {})
        regressions = [(c, v) for c, v in per_cls.items() if v < -0.03]
        if regressions:
            print("    Regressions (F1 drop > 0.03):")
            for cls, v in regressions:
                print(f"      ! {cls}: {v:+.3f}")
        print("    *** REGRESSION DETECTED ***" if reg else "    No significant regression detected.")
    else:
        print("\n  Comparison with previous run: N/A (first run or no baseline)")

    print("\n  Weak classes (f1 < 0.80 | recall < 0.75 | roc_auc < 0.85):")
    if state["weak_classes"]:
        for cls in state["weak_classes"]:
            print(f"    ! {cls}")
    else:
        print("    No weak classes identified.")

    section("ATTACK TYPES — CHARACTERISTICS SUMMARY")
    print_attack_descriptions(state.get("attack_descriptions", {}))

    section("PER-CLASS ANALYSIS")
    print(state.get("per_class_analysis", "(empty)"))

    section("CONFUSION MATRIX ANALYSIS")
    print(state.get("confusion_analysis", "(empty)"))

    recs = state.get("recommendations", {})
    section("RECOMMENDATIONS — Data generation")
    print_list(recs.get("data_generation", []))

    section("RECOMMENDATIONS — Model architecture/hyperparameters")
    print_list(recs.get("model", []))

    section("RECOMMENDATIONS — Training strategy")
    print_list(recs.get("training", []))

    decision = state.get("decision", "")
    if decision:
        section("DECISION")
        labels = {
            "SUGGEST_ONLY": "Suggestions only — model performance is acceptable.",
            "SCAN_FIRST":   "Codebase scanned — review proposed changes before retraining.",
            "RETRAIN":      "Retraining recommended — MCC below critical threshold.",
        }
        print(f"  => {decision}: {labels.get(decision, '')}")

    proposed = state.get("proposed_hyperparams", {})
    if proposed:
        section("PROPOSED HYPERPARAMETERS")
        reasoning = proposed.get("_reasoning", "")
        if reasoning:
            print(f"  Reasoning: {reasoning}\n")
        for param, value in proposed.items():
            if param == "_reasoning":
                continue
            print(f"    {param:<20} {value}")

    scanned = state.get("scanned_files", {})
    if scanned:
        print(f"\n  Scanned files ({len(scanned)}):")
        for fname, content in scanned.items():
            status = "[ERROR]" if content.startswith("[") else "[OK]"
            print(f"    {status} {fname}" + (f"  ({len(content)} chars)" if status == "[OK]" else ""))

    section("CONCLUSION")
    print(state.get("summary", "(empty)"))
    print()
    separator()


print(" Print helper funkcije učitane")

## [8] Pokretanje analize

Ovo je ekvivalent `run_test()` iz `langraph_test.py`. Pokreće ceo LangGraph graf i prikazuje rezultate.

> **Napomena o trajanju:** Groq je veoma brz (cloud inference), očekuj **15-45 sekundi** za kompletan run, u poređenju sa 2-5 minuta na lokalnom Ollama-u.

In [ ]:
import shutil
import time

async def run_analysis():
    p = METRICS_PATH

    print()
    separator()
    print("  DDoS LSTM — LangGraph Analyzer (Groq backend)")
    separator()

    # ─── Groq provera ──────────────────────────────────────────────────────
    print("\n[1/4] Checking Groq API...")
    try:
        test_resp = await ollama_generate("Say: ok", system="Reply with just 'ok'.")
        print(f"  Groq available | model: {GROQ_MODEL} | response: {test_resp[:30]}")
    except Exception as e:
        print(f"   Groq API not available: {e}")
        print("  Proveri GROQ_API_KEY u ćeliji [2].")
        return None

    # ─── Učitavanje metrika ─────────────────────────────────────────────────
    print(f"\n[2/4] Loading metrics from: {p}")
    if not p.exists():
        print(f"   File not found: {p}")
        print("  Uploaduj projekat na Drive i postavi tačnu putanju u ćeliji [2].")
        return None

    with open(p, "r") as f:
        metrics = json.load(f)

    print(f"  MCC score:     {metrics['mcc_score']:.4f}")
    print(f"  Classes:       {len(metrics['class_labels'])}")
    print(f"  Timestamp:     {metrics.get('timestamp', 'N/A')}")
    if "summary" in metrics:
        s = metrics["summary"]
        print(f"  Macro F1:      {s.get('macro_f1', 0):.4f}")
        print(f"  Total samples: {s.get('total_samples', 0):,}")

    # ─── Inicijalni state ───────────────────────────────────────────────────
    print("\n[3/4] Running LangGraph graph...")
    print("  Nodes: load_and_compare → load_attack_descriptions → analyze_per_class")
    print("         → analyze_confusion → synthesize → decision → [scan_codebase] → propose_hyperparams")
    print(f"  (Groq nodes brži od Ollama — ~15-45s ukupno)\n")

    initial_state = {
        "classification_report": metrics["classification_report"],
        "confusion_matrix":      metrics["confusion_matrix"],
        "mcc_score":             metrics["mcc_score"],
        "roc_auc_scores":        metrics["roc_auc_scores"],
        "class_labels":          metrics["class_labels"],
        "previous_metrics":      {},
        "metrics_delta":         {},
        "regression_detected":   False,
        "per_class_analysis":    "",
        "confusion_analysis":    "",
        "attack_descriptions":   {},
        "weak_classes":          [],
        "recommendations":       {},
        "summary":               "",
        "decision":              "",
        "scanned_files":         {},
        "proposed_hyperparams":  {},
        "human_confirmed":       False,
        "retrain_triggered":     False,
        "retrain_command":       "",
        "dataset_path":          "output/1d.csv",
        "model_name":            "",
    }

    t_start = time.time()
    try:
        final_state = await analyzer_graph.ainvoke(initial_state)
    except Exception as e:
        print(f"   ERROR in graph: {e}")
        import traceback
        traceback.print_exc()
        return None

    elapsed = time.time() - t_start

    # Backup metrika za sledeći run
    prev_path = p.parent / (p.stem + "_prev.json")
    shutil.copy(p, prev_path)
    print(f"  Metrics backed up → {prev_path}")
    print(f"[4/4] Graph finished in {elapsed:.1f}s")

    return final_state


# Pokreni analizu
final_state = await run_analysis()

if final_state:
    print_results(final_state)

## [9] Human Confirm & Retrain (opciono)

Ekvivalent `node_human_confirm` i `node_trigger_retrain` iz `langraph.py`.

**Van grafa su — kao i u originalnom kodu.**

> **Napomena za Colab:** `subprocess` za pokretanje `torch_nn.py` radi normalno ako je fajl dostupan na disku (u `PROJECT_ROOT`). Colab nema GPU timeout za kratke skripte, ali za dugo treniranje preporuči se pokretanje lokalno ili na Colab Pro sa A100.

Promeni `CONFIRM_RETRAIN = True` ako želiš da dopustiš retrain:

In [ ]:
import asyncio
import datetime
import shutil

# ─── Postavi na True da bi potvrdio retrain ──────────────────────────────────
CONFIRM_RETRAIN  = False           # True = pokreni retrain
DATASET_PATH     = "output/1d.csv" # putanja do CSV-a za treniranje
# ────────────────────────────────────────────────────────────────────────────

async def node_human_confirm_colab(state: dict, confirmed: bool, dataset_path: str) -> dict:
    """
    Colab verzija node_human_confirm — bez interaktivnog input()-a.
    Potvrda se kontroliše varijablama CONFIRM_RETRAIN i DATASET_PATH iznad.
    """
    proposed  = state.get("proposed_hyperparams", {})
    reasoning = proposed.get("_reasoning", "")

    print("\n" + "=" * 65)
    print("  RETRAIN CONFIRMATION")
    print("=" * 65)
    print(f"\n  Decision: {state.get('decision')} | MCC: {state['mcc_score']:.4f}")
    if reasoning:
        print(f"\n  Reasoning: {reasoning}")
    print("\n  Proposed hyperparameters:")
    for param, value in proposed.items():
        if param == "_reasoning":
            continue
        print(f"    {param:<20} {value}")

    print(f"\n  CONFIRM_RETRAIN = {confirmed}")
    if confirmed:
        print(f"  Dataset path:    {dataset_path}")
        print("  => Retrain will be triggered.")
    else:
        print("  => Retrain skipped. Set CONFIRM_RETRAIN = True to proceed.")

    return {"human_confirmed": confirmed, "dataset_path": dataset_path}


async def node_trigger_retrain_colab(state: dict) -> dict:
    """
    Colab verzija node_trigger_retrain.
    Identična logika — patchuje hyperparam.py i pokreće torch_nn.py subprocesom.
    """
    import re

    proposed = {k: v for k, v in state["proposed_hyperparams"].items() if k != "_reasoning"}
    timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    model_name = f"ddos_lstm_retrain_{timestamp}.pt"
    proposed["MODEL_NAME"] = model_name
    print(f"  New model will be saved as: {model_name}")

    # Patch hyperparam.py
    hp_path = _find_file_recursive("hyperparam.py", Path("."))
    if hp_path:
        content = hp_path.read_text(encoding="utf-8")
        backup  = hp_path.with_suffix(".py.bak")
        shutil.copy(hp_path, backup)
        print(f"  Backed up hyperparam.py → {backup}")

        for param, value in proposed.items():
            spec = _HYPERPARAMETER_SPACE.get(param)
            if spec is None:
                if isinstance(value, str):
                    pattern = rf'({re.escape(param)}\s*=\s*)["\'][^"\']*["\']'
                    content, n = re.subn(pattern, rf'\g<1>"{value}"', content)
                    if n: print(f"  Patched '{param}' → {value}")
                continue
            if spec["type"] == "choice":
                pattern, replacement = rf'("{param}"\s*:\s*)\[[^\]]*\]', rf'\g<1>[{value}]'
            else:
                pattern, replacement = rf'("{param}"\s*:\s*)\([^)]*\)', rf'\g<1>({value}, {value})'
            content, n = re.subn(pattern, replacement, content)
            if n: print(f"  Patched '{param}' → {value}")
        hp_path.write_text(content, encoding="utf-8")
    else:
        print("  hyperparam.py not found — using existing values.")

    # Pokretanje torch_nn.py
    torch_path = _find_file_recursive("torch_nn.py", Path("."))
    if torch_path is None:
        print("   torch_nn.py not found — cannot trigger retrain.")
        return {"retrain_triggered": False, "retrain_command": "NOT FOUND"}

    dataset_path = state.get("dataset_path") or "output/1d.csv"
    command = ["python", "-u", str(torch_path), dataset_path, model_name]
    command_str = " ".join(command)
    print(f"\n  Starting retrain: {command_str}")

    proc = await asyncio.create_subprocess_exec(
        *command,
        stdin=asyncio.subprocess.DEVNULL,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.STDOUT,
    )
    print(f"  PID: {proc.pid}\n")

    async for line in proc.stdout:
        print(f"  [train] {line.decode().rstrip()}", flush=True)

    await proc.wait()
    success = proc.returncode == 0
    print(f"\n  Retrain {'finished successfully' if success else 'finished with errors'} (exit code {proc.returncode}).")

    return {"retrain_triggered": success, "retrain_command": command_str}


# ─── Izvršavanje ─────────────────────────────────────────────────────────────

if final_state and final_state.get("proposed_hyperparams"):
    confirm_state = await node_human_confirm_colab(final_state, CONFIRM_RETRAIN, DATASET_PATH)
    final_state.update(confirm_state)

    if final_state.get("human_confirmed"):
        retrain_state = await node_trigger_retrain_colab(final_state)
        final_state.update(retrain_state)
        if final_state.get("retrain_triggered"):
            print(f"\n   Retrain started | command: {final_state.get('retrain_command', '')}")
    else:
        print("\n  Retrain skipped.")
elif final_state:
    print("  No hyperparameter proposal — decision was SUGGEST_ONLY.")

## [10] Čuvanje rezultata

Identično originalnom kodu — snima `langgraph_analysis.json` u isti folder kao `test_metrics.json`.

In [ ]:
if final_state:
    output_path = METRICS_PATH.parent / "langgraph_analysis.json"
    output = {
        "weak_classes":        final_state["weak_classes"],
        "per_class_analysis":  final_state["per_class_analysis"],
        "confusion_analysis":  final_state["confusion_analysis"],
        "recommendations":     final_state["recommendations"],
        "summary":             final_state["summary"],
        "decision":             final_state.get("decision", ""),
        "scanned_files":        list(final_state.get("scanned_files", {}).keys()),
        "proposed_hyperparams": final_state.get("proposed_hyperparams", {}),
        "metrics_delta":        final_state.get("metrics_delta", {}),
        "regression_detected":  final_state.get("regression_detected", False),
        "retrain_triggered":    final_state.get("retrain_triggered", False),
        "model_name":           final_state.get("model_name", ""),
        "source_metrics":       str(METRICS_PATH),
    }
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"   Analysis saved → {output_path}")
else:
    print("  Analiza nije završena — nema rezultata za čuvanje.")